Part 1 — Setup

1. Check GPU

In [1]:
import torch
torch.cuda.is_available()

True

2. Install dependencies

In [2]:
!pip install transformers datasets accelerate evaluate scikit-learn sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 25.9 MB/s eta 0:00:00


Part 2 — Load All Three Datasets

Load train, val, and test from disk

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile

with zipfile.ZipFile("/content/drive/MyDrive/csci316 project/train_dataset.zip", "r") as z:
    z.extractall("/content/train_dataset")

with zipfile.ZipFile("/content/drive/MyDrive/csci316 project/val_dataset.zip", "r") as z:
    z.extractall("/content/val_dataset")

with zipfile.ZipFile("/content/drive/MyDrive/csci316 project/test_dataset.zip", "r") as z:
    z.extractall("/content/test_dataset")

In [5]:
from datasets import load_from_disk

train_dataset = load_from_disk("/content/train_dataset")
val_dataset   = load_from_disk("/content/val_dataset")
test_dataset  = load_from_disk("/content/test_dataset")

print("Train:", train_dataset)
print("Val:  ", val_dataset)
print("Test: ", test_dataset)

Train: Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 2464
})
Val:   Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 818
})
Test:  Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 818
})


Part 3 — Inspect Data

Inspect training set

In [6]:
print(train_dataset.column_names)
print(train_dataset[0])
print("Label values in train:", set(train_dataset["label"]))

['label', 'input_ids', 'attention_mask']
{'label': tensor(0), 'input_ids': tensor([     0,  91010,    297,   3395,    186,   1884,    152,    615,  31910,
           648,    230, 144509,    906, 200317,    787,  96178,  38180,  35336,
           240, 125800,   1031,  54186,    754,  22963,    368,   1533,    716,
           477,  61591,  26894,    359,   9688,    304,  28521,    230,   1327,
         19931,   6861,  26059,      2,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      

 Part 4 — Load Model

Load XLM-RoBERTa for 3-class classification

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Part 5 — Metrics

Define metrics

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [10]:
import torch
import numpy as np
from torch import nn
from transformers import Trainer

label_array = np.array(train_dataset["label"])

class_counts = np.bincount(label_array)
total = len(label_array)
class_weights = total / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

Class counts: [ 594  778 1092]
Class weights: tensor([1.3827, 1.0557, 0.7521])


Part 6 — Training

Configure Training Arguments

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    logging_dir="./logs",

    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,


    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="epoch",
    logging_steps=1,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Part 7 — Trainer

 Create Trainer

In [12]:
from transformers import Trainer

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

Part 8 — Train


In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.480585,0.843507,0.823961,0.822190,0.827473,0.823961
2,0.356552,0.773188,0.850856,0.850906,0.851143,0.850856
3,0.258515,0.724761,0.849633,0.850346,0.852056,0.849633
4,0.159898,1.069870,0.842298,0.840396,0.843658,0.842298
5,0.126098,0.952037,0.859413,0.859029,0.859451,0.859413


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1540, training_loss=0.2763294715385932, metrics={'train_runtime': 1249.5395, 'train_samples_per_second': 9.86, 'train_steps_per_second': 1.232, 'total_flos': 810389326602240.0, 'train_loss': 0.2763294715385932, 'epoch': 5.0})

In [15]:
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("\n===== Test Set Results =====")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")


===== Test Set Results =====
  eval_loss: 0.9351
  eval_accuracy: 0.8704
  eval_f1: 0.8698
  eval_precision: 0.8696
  eval_recall: 0.8704
  eval_runtime: 5.5740
  eval_samples_per_second: 146.7520
  eval_steps_per_second: 18.4790
  epoch: 5.0000


In [16]:
trainer.save_model("/content/drive/MyDrive/csci316 project/full_finetuned_xlmr")
print("Model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


In [17]:
from transformers import AutoModelForSequenceClassification

loaded_model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/csci316 project/full_finetuned_xlmr"
)
print("Fine-tuned model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Fine-tuned model loaded successfully.
